# Ride-hailing dataset: preprocessing for ML/DL

Goal: turn `df_normalized.csv` into a fully numeric dataset with no missing values.

Steps
1. Load and inspect
2. Remove rows that are not real rides
3. Remove duplicate rows
4. Clean and consolidate categorical values
5. Fix the distance column
6. Date and time features
7. Location features
8. Choose the target and remove leaking columns
9. Drop redundant columns and one-hot encode
10. Scale numeric columns
11. Validate and save

## 0. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

INPUT  = "df_normalized.csv"
OUTPUT = "ml_ready.csv"

# Choose what you want to predict:
#   "fare_amount" -> regression
#   "status"      -> classification (ride outcome)
TARGET = "fare_amount"

## 1. Load and inspect

In [ ]:
df = pd.read_csv(INPUT)
print(df.shape)
df.head()

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nRows per source:")
print(df["source_dataset"].value_counts())

## 2. Remove rows that are not real rides

`delhi_notification_2023` and `aru_dto_2019` are official rate cards / fixed tariffs
(status is `official_rate_card`, `fixed`, `per_seat`, `per_day`; no date, no drop location,
constant ratings). They are not trips, so they would only add noise. Removing them also
removes every NaN in `date`, `weekday` and `drop_location`.

In [ ]:
NON_TRIP_SOURCES = ["delhi_notification_2023", "aru_dto_2019"]

mask = df["source_dataset"].isin(NON_TRIP_SOURCES)
print("Dropping", mask.sum(), "rate-card / tariff rows")
df = df.loc[~mask].copy()
df.shape

## 3. Remove duplicate rows

All duplicates are in `ncr_events`. There is no ID column, but with date, hour, locations,
distance and fare all identical, these are almost certainly repeated records and not
coincidences.

In [ ]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Dropped", n_before - len(df), "duplicate rows ->", df.shape)

## 4. Clean and consolidate categorical values

* `payment_method` has 11 labels that overlap (card / credit card / debit card, ...).
  Group them into cash, card, upi, wallet.
* `status`: `success` and `completed` mean the same thing (different source files).

In [ ]:
for c in ["source_dataset", "vehicle_type", "payment_method", "status", "state", "season", "weather"]:
    df[c] = df[c].astype(str).str.strip().str.lower()

PAYMENT_MAP = {
    "cash": "cash",
    "card": "card", "credit card": "card", "debit card": "card",
    "upi": "upi", "gpay": "upi", "qr scan": "upi",
    "wallet": "wallet", "uber wallet": "wallet", "paytm": "wallet", "amazon pay": "wallet",
}
STATUS_MAP = {
    "success": "completed", "completed": "completed",
    "incomplete": "incomplete",
    "canceled by customer": "canceled by customer",
    "canceled by driver": "canceled by driver",
    "driver not found": "driver not found",
}

df["payment_group"] = df["payment_method"].map(PAYMENT_MAP)
df["status"] = df["status"].map(STATUS_MAP)

assert df["payment_group"].notna().all(), "unmapped payment_method"
assert df["status"].notna().all(), "unmapped status"
print(df["payment_group"].value_counts(), "\n")
print(df["status"].value_counts())

## 5. Fix the distance column

`distance_km == 0` only happens for cancelled bookings, where the ride never started. The
value is really "not recorded", so treat it as missing, keep a flag, and fill with the
median distance of the same source.

In [ ]:
df["distance_missing"] = (df["distance_km"] <= 0).astype(int)
df.loc[df["distance_km"] <= 0, "distance_km"] = np.nan
df["distance_km"] = df["distance_km"].fillna(
    df.groupby("source_dataset")["distance_km"].transform("median")
)
print("Missing-distance rows flagged:", df["distance_missing"].sum())
print("NaN left in distance_km:", df["distance_km"].isna().sum())

## 6. Date and time features

* `date` comes in two formats (`YYYY-MM-DD` and `YYYY-MM-DD HH:MM:SS`), so parse the first 10 characters.
* `hour` is missing for about 36% of rows. `phase_of_day` is a fixed function of hour
  (midnight 0-3, before_sunrise 4-5, morning 6-11, afternoon 12-15, evening 16-18, night 19-23),
  so fill the hour with the typical hour of that phase and keep `is_phase_imputed`.
* Cyclic values (hour, month, day, weekday) are encoded as sin/cos so that 23:00 is close to 00:00.

In [ ]:
date = pd.to_datetime(df["date"].astype(str).str[:10], format="%Y-%m-%d")
df["year"] = date.dt.year
df["month"] = date.dt.month
df["day"] = date.dt.day
df["dow"] = date.dt.dayofweek          # 0 = Monday
df["is_weekend"] = (df["dow"] >= 5).astype(int)

# impute hour from phase_of_day
df["is_phase_imputed"] = df["is_phase_imputed"].astype(int)
phase_hour = df.loc[df["hour"].notna()].groupby("phase_of_day")["hour"].median().round()
print(phase_hour)
df["hour"] = df["hour"].fillna(df["phase_of_day"].map(phase_hour))
assert df["hour"].notna().all()

def add_cyclical(data, col, period):
    data[f"{col}_sin"] = np.sin(2 * np.pi * data[col] / period)
    data[f"{col}_cos"] = np.cos(2 * np.pi * data[col] / period)

add_cyclical(df, "hour", 24)
add_cyclical(df, "month", 12)
add_cyclical(df, "day", 31)
add_cyclical(df, "dow", 7)

## 7. Location features

There are about 12.8k unique pickup locations. One-hot encoding would create about 13k
columns, so use the log of how often each location appears, plus a flag for pickup == drop.

In [ ]:
df["pickup_location"] = df["pickup_location"].astype(str).str.strip().str.lower()
df["drop_location"]   = df["drop_location"].astype(str).str.strip().str.lower()

counts = pd.concat([df["pickup_location"], df["drop_location"]]).value_counts()
df["pickup_freq"] = np.log1p(df["pickup_location"].map(counts))
df["drop_freq"]   = np.log1p(df["drop_location"].map(counts))
df["same_pickup_drop"] = (df["pickup_location"] == df["drop_location"]).astype(int)

## 8. Choose the target and remove leaking columns

For `status` prediction, three things leak the answer, so they are removed:
* ratings are filled with a constant 4.2 for every non-completed ride
* `distance_km` is 0 for every cancelled booking
* `weather == "unknown"` appears only on cancelled bookings

In [ ]:
STATUS_CODES = {
    "completed": 0, "incomplete": 1, "canceled by customer": 2,
    "canceled by driver": 3, "driver not found": 4,
}
LEAKY_FOR_STATUS = ["driver_rating", "customer_rating", "distance_km", "distance_missing", "weather"]

if TARGET == "fare_amount":
    y = df["fare_amount"].astype(float)
    drop_cols = ["fare_amount"]
else:
    y = df["status"].map(STATUS_CODES).astype(int)
    drop_cols = list(LEAKY_FOR_STATUS)

y.describe() if TARGET == "fare_amount" else y.value_counts()

## 9. Drop redundant columns and one-hot encode

* raw date/time columns are replaced by the sin/cos features
* `phase_of_day` is fully determined by hour
* `weekday` duplicates `dow`
* `state` is fully determined by `source_dataset`
* location names are replaced by the frequency features

In [ ]:
drop_cols += [
    "date", "hour", "month", "day", "dow",
    "phase_of_day", "weekday",
    "payment_method", "status",
    "state",
    "pickup_location", "drop_location",
]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

ONE_HOT_COLS = ["source_dataset", "vehicle_type", "payment_group", "season", "weather"]
ohe_cols = [c for c in ONE_HOT_COLS if c in X.columns]
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True, dtype=int)
print(X.shape)
X.head()

## 10. Scale numeric columns

Standardise the continuous columns (mean 0, std 1). Binary, one-hot and sin/cos columns are left as they are.
The target is NOT scaled.

In [ ]:
to_scale = [c for c in ["distance_km", "fare_amount", "driver_rating", "customer_rating",
                        "year", "pickup_freq", "drop_freq"] if c in X.columns]
print("Scaling:", to_scale)

scaler = StandardScaler()
X[to_scale] = scaler.fit_transform(X[to_scale])
X[to_scale].describe().round(2)

## 11. Validate and save

In [ ]:
out = X.copy()
out["target"] = y.values

assert not out.isna().any().any(), "NaNs remain"
assert np.isfinite(out.to_numpy(dtype=float)).all(), "inf remains"
assert all(pd.api.types.is_numeric_dtype(t) for t in out.dtypes), "non-numeric column remains"

out.to_csv(OUTPUT, index=False, float_format="%.6g")
print(out.shape, "->", OUTPUT)
out.head()